In [1]:
import os
import torch
import numpy as np
import pandas as pd
import nibabel as nib
from torch.utils.data import Dataset

OASIS_DIR = "../../data/oasis"

class OASISDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = nib.load(row["scan_path"])
        volume = img.get_fdata().squeeze()

        mid = volume.shape[2] // 2
        slice_stack = volume[:, :, mid-1:mid+2]
        slice_stack = np.transpose(slice_stack, (2, 0, 1))

        slice_stack = slice_stack.astype(np.float32)
        slice_stack = (slice_stack - slice_stack.min()) / (slice_stack.max() - slice_stack.min() + 1e-8)

        tensor = torch.from_numpy(slice_stack)
        if self.transform:
            tensor = self.transform(tensor)

        label = torch.tensor(row["label"], dtype=torch.long)
        return tensor, label

In [2]:
from torch.utils.data import DataLoader

BATCH_SIZE = 16

train_ds = OASISDataset(f"{OASIS_DIR}/splits/train.csv")
val_ds = OASISDataset(f"{OASIS_DIR}/splits/val.csv")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}")

Train batches: 11  |  Val batches: 3


In [3]:
# ResNet-50 with ImageNet transfer learning
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

# Replace the final classification layer: ImageNet has 1000 classes, we need 2
# (Nondemented / Demented). Everything before this layer keeps its pretrained weights.
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

print("Model ready. Final layer:", model.fc)

Using device: mps
Model ready. Final layer: Linear(in_features=2048, out_features=2, bias=True)


In [4]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss, correct, total = 0, 0, 0

    with torch.set_grad_enabled(training):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            if training:
                optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            if training:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total

# Sanity-check run: 1 epoch, confirms the pipeline works end to end on your Mac.
# Full training (many epochs, multiple seeds) happens on Kaggle, not here.
NUM_EPOCHS = 1
for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = run_epoch(train_loader, training=True)
    val_loss, val_acc = run_epoch(val_loader, training=False)
    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
          f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

Epoch 1: train_loss=0.6761 train_acc=0.5854 | val_loss=0.6694 val_acc=0.5429
